# C1.4 Embeddings, Semantic Search and Mini-RAG

This notebook follows the same teaching flow as the draft in C1.4.ipynb, but tightens the explanation, adds a local 25-document corpus, and includes the missing lab-ready pieces for semantic search and mini-RAG.

## Learning goals

- Build intuition for embeddings and cosine similarity without a heavy math detour.
- Compare sparse search, dense search, FAISS, and ChromaDB on the same corpus.
- Assemble a mini-RAG pipeline that retrieves context before generation.
- Use the provided 10-query lab set to compare baseline and grounded answers.

In [1]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    import faiss
    FAISS_AVAILABLE = True
except Exception as exc:
    faiss = None
    FAISS_AVAILABLE = False
    faiss_error = exc

try:
    from sentence_transformers import SentenceTransformer
    ST_AVAILABLE = True
except Exception as exc:
    SentenceTransformer = None
    ST_AVAILABLE = False
    st_error = exc

## Embeddings fundamentals

Embeddings are numeric representations of text. The model turns a piece of text into a vector so that texts with similar meaning end up close together in vector space.

That is the key idea behind semantic similarity: you are not matching exact keywords only, you are comparing meaning.

Cosine similarity is a simple way to compare two vectors by checking whether they point in a similar direction. For embeddings, a higher cosine similarity usually means the texts are more related.

In [2]:
tiny_docs = [
    "Embeddings turn text into vectors that capture meaning.",
    "Cosine similarity measures how close two vectors point in the same direction.",
    "Football is a popular sport played by two teams."
]

tiny_query = "How do vectors help compare meaning?"
tiny_vectorizer = TfidfVectorizer()
tiny_matrix = tiny_vectorizer.fit_transform(tiny_docs + [tiny_query])
tiny_query_vec = tiny_matrix[-1]
tiny_doc_vecs = tiny_matrix[:-1]
tiny_scores = cosine_similarity(tiny_query_vec, tiny_doc_vecs)[0]
tiny_ranked = tiny_scores.argsort()[::-1]

print("Query:", tiny_query)
for idx in tiny_ranked:
    print(f"- {tiny_docs[idx]} (score: {tiny_scores[idx]:.3f})")

Query: How do vectors help compare meaning?
- Embeddings turn text into vectors that capture meaning. (score: 0.180)
- Cosine similarity measures how close two vectors point in the same direction. (score: 0.146)
- Football is a popular sport played by two teams. (score: 0.000)


Interpretation: The query is closest to the embedding and cosine-similarity descriptions, while the unrelated sports sentence scores near zero. That is the core intuition behind semantic search: similar meanings produce nearby vectors.

### Text embedding models available via API

The hands-on demo below uses a local SentenceTransformer model so learners can run it offline, but the same workflow works with API-based embeddings.

Common API options include:

- OpenAI embedding models such as `text-embedding-3-small` and `text-embedding-3-large`.
- Cohere embedding models such as `embed-english-v3.0` and `embed-multilingual-v3.0`.
- Other hosted embedding providers that return dense vectors for similarity search.

The important pattern is the same: text goes in, a vector comes out, and that vector can be indexed for search.

## Vector search and stores

This module uses a curated local corpus stored in `data/C1.4 corpus/docs.jsonl`. The file already contains 25 short documents, so learners can focus on search and retrieval rather than data collection.

We will use the same corpus for both sparse and dense search, then extend the dense retriever into a mini-RAG pipeline.

In [3]:
corpus_path = Path("data") / "C1.4 corpus" / "docs.jsonl"
if not corpus_path.exists():
    raise FileNotFoundError(f"Missing corpus file: {corpus_path}")

corpus_records = [
    json.loads(line)
    for line in corpus_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
corpus_df = pd.DataFrame(corpus_records)
documents = corpus_df["text"].tolist()
titles = corpus_df["title"].tolist()
document_ids = corpus_df["id"].tolist()

print(f"Loaded {len(corpus_df)} documents")
display(corpus_df[["id", "title", "tags"]].head())

Loaded 25 documents


,id,title,tags
0,c1_01,Embeddings make meaning searchable,"[embeddings, semantic-search]"
1,c1_02,Tokenization before embedding,"[tokenization, embeddings]"
2,c1_03,Cosine similarity intuition,"[cosine-similarity, embeddings]"
3,c1_04,OpenAI style embedding APIs,"[api, embeddings]"
4,c1_05,SentenceTransformer models,"[sentence-transformers, open-source]"


Interpretation: The corpus contains 25 curated documents, which is enough to demonstrate semantic search and mini-RAG without making learners collect data first. The preview shows a mix of embeddings, retrieval, RAG, and general knowledge topics.

In [4]:
sparse_vectorizer = TfidfVectorizer(stop_words="english")
sparse_doc_vectors = sparse_vectorizer.fit_transform(documents)

def sparse_search(query, k=3):
    query_vec = sparse_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, sparse_doc_vectors)[0]
    ranked_indices = scores.argsort()[-k:][::-1]
    results = []
    for idx in ranked_indices:
        results.append({
            "id": document_ids[idx],
            "title": titles[idx],
            "text": documents[idx],
            "score": float(scores[idx])
        })
    return results

sample_query = "What is RAG and why does it help?"
for item in sparse_search(sample_query, k=3):
    print(f"{item['title']} | score={item['score']:.3f}")
    print(item['text'])
    print()

Top-k retrieval | score=0.266
Top-k retrieval returns the k most similar chunks for a query. It is simple, fast, and common in semantic search demos and RAG pipelines.

Hallucination is unsupported output | score=0.214
Hallucination happens when a model gives a confident answer that is not supported by the facts. In RAG systems this risk is lower because the model can read the retrieved context first.

Mini-RAG pipeline steps | score=0.209
A mini-RAG pipeline usually follows three steps: retrieve the top chunks, augment the prompt with those chunks, and generate the response. The workflow is simple but powerful.



Interpretation: Sparse TF-IDF search mostly follows keyword overlap, so the RAG-related terms rise to the top. This is useful as a baseline, but it does not capture meaning as well as dense retrieval.

In [5]:
if not ST_AVAILABLE or not FAISS_AVAILABLE:
    raise RuntimeError("Install sentence-transformers and faiss-cpu to run the dense FAISS demo.")

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = np.asarray(embed_model.encode(documents, convert_to_numpy=True), dtype="float32")
faiss.normalize_L2(doc_embeddings)
dimension = doc_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(doc_embeddings)

def dense_search(query, k=3):
    query_vec = np.asarray(embed_model.encode([query], convert_to_numpy=True), dtype="float32")
    faiss.normalize_L2(query_vec)
    scores, indices = faiss_index.search(query_vec, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "id": document_ids[idx],
            "title": titles[idx],
            "text": documents[idx],
            "score": float(score)
        })
    return results

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\AKASH.O\PYTHON\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Akash\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Interpretation: The embedding model downloaded successfully and the FAISS index was built from normalized vectors. The Windows and Hugging Face warnings are environmental, not functional, so they do not block the semantic search demo.

In [6]:
dense_queries = [
    "How does semantic search work?",
    "Why does chunk overlap matter?",
    "How does RAG reduce hallucination?"
]

for query in dense_queries:
    print("Query:", query)
    for item in dense_search(query, k=3):
        print(f"- {item['title']} | score={item['score']:.3f}")
    print()

Query: How does semantic search work?
- Embeddings make meaning searchable | score=0.565
- Top-k retrieval | score=0.513
- Query rewriting | score=0.489

Query: Why does chunk overlap matter?
- Chunk overlap preserves context | score=0.767
- Chunking improves retrieval | score=0.463
- Top-k retrieval | score=0.233

Query: How does RAG reduce hallucination?
- Hallucination is unsupported output | score=0.676
- Mini-RAG pipeline steps | score=0.193
- Grounding reduces hallucination | score=0.152



Interpretation: Dense FAISS search retrieves meaning-related passages instead of only keyword matches. For the sample queries, it surfaces the most relevant conceptual neighbors such as semantic search, chunk overlap, and hallucination.

### FAISS and ChromaDB

FAISS is the fast nearest-neighbor index used in the dense search demo above. ChromaDB gives you a vector store with documents, metadata, and querying in one place.

The key teaching point is that both systems support the same basic workflow: embed the corpus, index the vectors, query with a new embedding, and rank the nearest matches.

In [ ]:
try:
    import chromadb
    CHROMA_AVAILABLE = True
except Exception as exc:
    chromadb = None
    CHROMA_AVAILABLE = False
    chroma_error = exc

if CHROMA_AVAILABLE:
    chroma_client = chromadb.Client()
    chroma_collection = chroma_client.get_or_create_collection(name="c1_4_demo")
    chroma_collection.add(
        ids=document_ids,
        documents=documents,
        embeddings=doc_embeddings.tolist(),
        metadatas=[{"title": title, "id": doc_id} for title, doc_id in zip(titles, document_ids)]
    )

    def chroma_search(query, k=3):
        query_vec = np.asarray(embed_model.encode([query], convert_to_numpy=True), dtype="float32")
        faiss.normalize_L2(query_vec)
        result = chroma_collection.query(query_embeddings=query_vec.tolist(), n_results=k)
        items = []
        for doc_id, title, text, distance in zip(
            result["ids"][0],
            result["metadatas"][0],
            result["documents"][0],
            result["distances"][0],
        ):
            items.append({
                "id": doc_id,
                "title": title["title"],
                "text": text,
                "score": float(distance)
            })
        return items

    print("ChromaDB index ready.")
    for item in chroma_search("What is semantic search?", k=3):
        print(f"{item['title']} | distance={item['score']:.3f}")
else:
    print(f"ChromaDB is not available: {chroma_error}")

## Mini-RAG pipeline

A mini-RAG system does three things:

1. Retrieve the most relevant chunks.
2. Augment the prompt with the retrieved context.
3. Generate a grounded answer from the language model.

The notebook uses the dense FAISS retriever for retrieval because it already gives us semantic matching over the provided corpus.

In [7]:
def build_context(results):
    blocks = []
    for item in results:
        blocks.append(f"[Doc: {item['title']} | score={item['score']:.3f}]\n{item['text']}")
    return "\n\n".join(blocks)

def build_prompt(query, context):
    return f"""You are a helpful assistant for an AI engineering course.
Use the context to answer the question. If the context is not enough, say so clearly.

Context:
{context}

Question:
{query}

Answer:
"""

def retrieve(query, k=3):
    return dense_search(query, k=k)

def rag_pipeline(query, k=3):
    results = retrieve(query, k=k)
    context = build_context(results)
    prompt = build_prompt(query, context)
    return {
        "query": query,
        "results": results,
        "context": context,
        "prompt": prompt,
    }

In [8]:
try:
    from langchain_groq.chat_models import ChatGroq
    GROQ_AVAILABLE = True
except Exception as exc:
    ChatGroq = None
    GROQ_AVAILABLE = False
    groq_error = exc

groq_token = os.getenv("CHAT_GROQ_API_KEY")
llm = None
if GROQ_AVAILABLE and groq_token:
    llm = ChatGroq(groq_api_key=groq_token, model_name="llama-3.1-8b-instant")

def non_rag_answer(query):
    if llm is None:
        return "Set CHAT_GROQ_API_KEY to run the live baseline answer."
    return llm.invoke(query).content

def rag_answer(query, k=3):
    if llm is None:
        return "Set CHAT_GROQ_API_KEY to run the live RAG answer."
    payload = rag_pipeline(query, k=k)
    response = llm.invoke(payload["prompt"])
    return response.content

In [9]:
query = "Which concepts in the corpus explain why semantic search and RAG are useful? Answer in 50 words."

baseline = non_rag_answer(query)
grounded = rag_answer(query)

print("--- NON-RAG baseline ---")
print(baseline)
print()
print("--- RAG answer ---")
print(grounded)

--- NON-RAG baseline ---
In the context of corpus-based knowledge, the following concepts explain why semantic search and Retrieval-Augmented Generation (RAG) are useful:

1. **Entity Disambiguation**: Semantic search resolves ambiguity by understanding context and intent, retrieving relevant information.
2. **Knowledge Graph Embeddings**: RAG uses embeddings to capture complex relationships between entities, enabling more accurate and relevant retrieval.
3. **Semantic Similarity**: Both techniques leverage semantic similarity to rank results based on relevance, rather than word overlap.
4. **Contextual Understanding**: RAG's use of context and intent helps to capture nuances and subtleties in user queries.
5. **Knowledge Retrieval and Generation**: RAG combines retrieval and generation to provide more comprehensive and accurate responses.

--- RAG answer ---
Based on the provided context, the concepts that explain why semantic search and RAG are useful are:

Embeddings, which enable t

Interpretation: This is the mini-RAG comparison point. The baseline answer relies only on the model's prior knowledge, while the RAG answer is generated after retrieval and should stay more grounded in the corpus context.

## Lab: Semantic Search and Mini-RAG

Use the 10 test queries below to compare retrieval-grounded answers against the baseline on the same question. The corpus is already provided, so learners should not spend lab time collecting documents.

For each query, inspect:

- the top retrieved chunk(s),
- the baseline answer,
- the RAG answer,
- whether the RAG answer is more grounded, relevant, and complete.

If the API key is available, run both answer paths. If not, use the retrieval output to explain what context would be injected into the prompt.

In [10]:
test_queries = [
    "What are embeddings and why do similar texts end up close together?",
    "How does cosine similarity help compare text vectors?",
    "What is the difference between sparse search and dense search?",
    "Why do we chunk long documents before retrieval?",
    "What is FAISS used for?",
    "How does ChromaDB store and query vectors?",
    "Why does RAG reduce hallucination?",
    "What is the role of query rewriting in search?",
    "What is the difference between recall and precision?",
    "When would you prefer an API embedding model over a local model?"
]

def compare_query(query, k=3):
    retrieved = retrieve(query, k=k)
    return {
        "query": query,
        "retrieved": [{"title": item["title"], "score": round(item["score"], 3)} for item in retrieved],
        "baseline": non_rag_answer(query),
        "rag": rag_answer(query, k=k)
    }

for query in test_queries:
    print("Query:", query)
    for item in retrieve(query, k=2):
        print(f"  - {item['title']} | score={item['score']:.3f}")
    print()

Query: What are embeddings and why do similar texts end up close together?
  - Cosine similarity intuition | score=0.643
  - Embeddings make meaning searchable | score=0.607

Query: How does cosine similarity help compare text vectors?
  - Cosine similarity intuition | score=0.862
  - Embeddings make meaning searchable | score=0.447

Query: What is the difference between sparse search and dense search?
  - Top-k retrieval | score=0.393
  - SentenceTransformer models | score=0.381

Query: Why do we chunk long documents before retrieval?
  - Chunking improves retrieval | score=0.782
  - Chunk overlap preserves context | score=0.498

Query: What is FAISS used for?
  - FAISS is a vector index | score=0.454
  - Paris is a major city | score=0.247

Query: How does ChromaDB store and query vectors?
  - ChromaDB stores embeddings and metadata | score=0.739
  - Bi-encoder and cross-encoder | score=0.407

Query: Why does RAG reduce hallucination?
  - Hallucination is unsupported output | score=0

Interpretation: The 10 queries cover the main course ideas from embeddings through RAG. Each printed retrieval list shows which passages would be injected into the prompt before generation, which makes it easy to compare grounding quality later.